# 2강. 임베딩 벡터는 어떻게 학습되는가?

오늘 목표는 하나입니다.

> **랜덤으로 시작한 embedding matrix가 loss와 gradient descent를 통해 어떻게 바뀌는지 확인한다.**

PyTorch의 `nn.Embedding`은 정수 index를 받아 해당 row vector를 반환하는 lookup table입니다. 즉, `input_ids`가 들어오면 embedding matrix의 row들이 선택됩니다. 이 embedding matrix는 일반적인 neural network parameter처럼 학습될 수 있습니다. PyTorch의 `autograd`는 forward 계산 그래프를 추적하고, `loss.backward()`를 통해 gradient를 계산하는 자동 미분 엔진입니다.

## 2-1. 오늘 만들 모델

아주 작은 character-level next-token predictor를 만들겠습니다.

예를 들어 문장이:

`"hello"`

라면 학습 데이터는 이렇게 만듭니다.

```text
h → e
e → l
l → l
l → o
```

즉, 현재 문자를 보고 다음 문자를 예측하는 문제입니다.

모델 구조는 다음과 같습니다.

```text
input character id
→ embedding lookup
→ linear layer
→ logits
→ cross entropy loss
```

아직 Transformer는 안 씁니다. 오늘은 embedding이 학습되는 원리만 봅니다.

## 2-2. 데이터 만들기

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

text = "hello embedding model"

# vocabulary
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

ids = encode(text)

print("chars:", chars)
print("ids:", ids)

chars: [' ', 'b', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o']
ids: [5, 3, 7, 7, 10, 0, 3, 8, 1, 3, 2, 2, 6, 9, 4, 0, 8, 10, 2, 3, 7]


In [2]:
#이제 x → y 쌍을 만듭니다.
x_ids = ids[:-1]
y_ids = ids[1:]

x = torch.tensor(x_ids, dtype=torch.long)
y = torch.tensor(y_ids, dtype=torch.long)

print("x:", x)
print("y:", y)

for xi, yi in zip(x, y):
    print(itos[xi.item()], "->", itos[yi.item()])

x: tensor([ 5,  3,  7,  7, 10,  0,  3,  8,  1,  3,  2,  2,  6,  9,  4,  0,  8, 10,
         2,  3])
y: tensor([ 3,  7,  7, 10,  0,  3,  8,  1,  3,  2,  2,  6,  9,  4,  0,  8, 10,  2,
         3,  7])
h -> e
e -> l
l -> l
l -> o
o ->  
  -> e
e -> m
m -> b
b -> e
e -> d
d -> d
d -> i
i -> n
n -> g
g ->  
  -> m
m -> o
o -> d
d -> e
e -> l


## 2-3. 모델 정의

In [3]:
class BigramEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        # input_ids: [sequence_length]
        emb = self.embedding(input_ids)
        # emb: [sequence_length, embedding_dim]

        logits = self.linear(emb)
        # logits: [sequence_length, vocab_size]

        return logits

여기서 shape를 잘 봐야 합니다.

- `input_ids.shape = [sequence_length]`
- `embedding(input_ids).shape = [sequence_length, embedding_dim]`
- `linear(embedding).shape = [sequence_length, vocab_size]`

왜 마지막이 `vocab_size`일까요?

다음 문자가 vocabulary 안의 어떤 문자일지 맞혀야 하기 때문입니다.

예를 들어 vocab이 12개 문자라면, 각 위치마다 12개 후보에 대한 score를 출력해야 합니다.

# 2-4. 학습 전 embedding 확인

In [5]:
vocab_size = len(chars)
embedding_dim = 8

model = BigramEmbeddingModel(vocab_size, embedding_dim)

target_char = "e"
target_id = stoi[target_char]

before = model.embedding.weight[target_id].detach().clone()

print("target char:", target_char)
print("before training embedding:")
print(before)

target char: e
before training embedding:
tensor([-0.0230, -0.1712,  0.2872,  0.2895, -0.1445,  0.0489, -2.1650,  0.1222])


여기서 `model.embedding.weight`가 바로 embedding matrix입니다.

`model.embedding.weight.shape = [vocab_size, embedding_dim]`

## 2-5. Loss 계산

In [ ]:
logits = model(x)

print("logits shape:", logits.shape)
print("target y shape:", y.shape)

loss = F.cross_entropy(logits, y)
print("initial loss:", loss.item())

logits shape: torch.Size([20, 11])
target y shape: torch.Size([20])
initial loss: 2.427919387817383


여기서 중요한 점:

- `logits.shape = [sequence_length, vocab_size]`
- `y.shape      = [sequence_length]`

각 위치마다 다음 문자 정답이 하나씩 있습니다.

**예:**

```text
input:  h    e    l    l
target: e    l    l    o
```

모델은 각 input token마다 다음 token을 예측합니다.

## 2-6. 학습 루프

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

for step in range(300):
    logits = model(x)
    loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step:03d} | loss {loss.item():.4f}")

step 000 | loss 2.4279
step 050 | loss 0.6522
step 100 | loss 0.6510
step 150 | loss 0.6508
step 200 | loss 0.6507
step 250 | loss 0.6506


이 부분이 핵심입니다.

`loss.backward()`

는 loss를 줄이기 위해 각 parameter가 어느 방향으로 바뀌어야 하는지 gradient를 계산합니다.

`optimizer.step()`

은 그 gradient를 사용해서 parameter를 실제로 업데이트합니다.

업데이트되는 parameter는 크게 두 종류입니다.

1. `embedding.weight`
2. `linear.weight`, `linear.bias`

즉, embedding vector도 학습됩니다.

## 2-7. 학습 후 embedding 확인

In [10]:
after = model.embedding.weight[target_id].detach().clone()

print("target char:", target_char)
print("before:")
print(before)

print("after:")
print(after)

print("difference:")
print(after - before)

print("L2 change:", torch.norm(after - before).item())

target char: e
before:
tensor([-0.0230, -0.1712,  0.2872,  0.2895, -0.1445,  0.0489, -2.1650,  0.1222])
after:
tensor([-1.5304, -0.4654, -0.0508,  0.5640, -0.3252,  1.7220, -3.0409, -1.0127])
difference:
tensor([-1.5075, -0.2942, -0.3379,  0.2745, -0.1807,  1.6731, -0.8760, -1.1349])
L2 change: 2.7268383502960205


여기서 L2 change가 0보다 크면, 해당 token의 embedding vector가 바뀐 것입니다.

즉:

```text
랜덤 embedding vector
→ loss 계산
→ gradient 계산
→ optimizer update
→ embedding vector 변화
```

가 실제로 일어난 것입니다.

## 2-8. 학습된 모델로 다음 문자 예측해보기

In [11]:
def predict_next_char(ch):
    input_id = torch.tensor([stoi[ch]], dtype=torch.long)

    logits = model(input_id)
    probs = F.softmax(logits[0], dim=-1)

    next_id = torch.argmax(probs).item()

    return itos[next_id], probs[next_id].item()

for ch in chars:
    pred, prob = predict_next_char(ch)
    print(f"{repr(ch)} -> {repr(pred)} | prob={prob:.3f}")

' ' -> 'e' | prob=0.500
'b' -> 'e' | prob=0.999
'd' -> 'i' | prob=0.333
'e' -> 'l' | prob=0.500
'g' -> ' ' | prob=0.999
'h' -> 'e' | prob=1.000
'i' -> 'n' | prob=0.999
'l' -> 'o' | prob=0.500
'm' -> 'o' | prob=0.500
'n' -> 'g' | prob=0.999
'o' -> ' ' | prob=0.500


예를 들어 `"h"` 다음에는 `"e"`가 자주 나오므로, 모델이 잘 학습되면:

`'h' -> 'e'`

처럼 예측할 수 있습니다.

> 다만 corpus가 너무 작기 때문에 “언어를 이해했다”고 보면 안 됩니다.
> 오늘 목적은 의미 이해가 아니라 embedding parameter가 학습되는 과정 확인입니다.

## 2-9. 핵심 개념 정리

### 1. 임베딩은 처음에는 랜덤이다
`nn.Embedding(vocab_size, embedding_dim)`
을 만들면 embedding matrix는 학습 가능한 parameter로 초기화됩니다.
처음에는 의미가 없습니다.

### 2. token id는 row를 선택한다
`embedding(torch.tensor([3]))`
은 embedding matrix에서 3번 row를 가져옵니다.
`E[3]`입니다.

### 3. 선택된 embedding vector가 예측에 사용된다
```text
token id
→ embedding vector
→ linear layer
→ logits
→ loss
```

### 4. loss가 크면 gradient가 생긴다
모델 예측이 틀리면 loss가 커지고, 그 loss를 줄이는 방향으로 gradient가 계산됩니다.

### 5. optimizer가 embedding vector를 업데이트한다
`optimizer.step()`이 호출되면 `embedding.weight`도 바뀝니다.

결국 embedding vector는 학습 과정에서 점점 task에 맞게 조정됩니다.

---

## 💡 오늘 가장 중요한 문장
> **embedding vector는 사람이 의미를 직접 넣는 것이 아니라, loss를 줄이는 방향으로 반복 업데이트되면서 의미 있는 구조를 획득한다.**

*단, 오늘 예제는 너무 작아서 “의미”보다는 “다음 문자 예측에 유리한 구조”가 생긴다고 보는 게 정확합니다.*